In [1]:
import polars as pl
import pandas as pd
from lightgbm import LGBMClassifier

In [4]:
df = pl.scan_parquet('../dataset/final_train_data/final_data.parquet')

print(df.columns)

# df.select('S_2').head()

['customer_ID', 'target', 'P_2_mean', 'D_39_mean', 'B_1_mean', 'B_2_mean', 'R_1_mean', 'S_3_mean', 'D_41_mean', 'B_3_mean', 'D_42_mean', 'D_43_mean', 'D_44_mean', 'B_4_mean', 'D_45_mean', 'B_5_mean', 'R_2_mean', 'D_46_mean', 'D_47_mean', 'D_48_mean', 'D_49_mean', 'B_6_mean', 'B_7_mean', 'B_8_mean', 'D_50_mean', 'D_51_mean', 'B_9_mean', 'R_3_mean', 'D_52_mean', 'P_3_mean', 'B_10_mean', 'D_53_mean', 'S_5_mean', 'B_11_mean', 'S_6_mean', 'D_54_mean', 'R_4_mean', 'S_7_mean', 'B_12_mean', 'S_8_mean', 'D_55_mean', 'D_56_mean', 'B_13_mean', 'R_5_mean', 'D_58_mean', 'S_9_mean', 'B_14_mean', 'D_59_mean', 'D_60_mean', 'D_61_mean', 'B_15_mean', 'S_11_mean', 'D_62_mean', 'D_63_mean', 'D_64_mean', 'D_65_mean', 'B_16_mean', 'B_17_mean', 'B_18_mean', 'B_19_mean', 'D_66_mean', 'B_20_mean', 'D_68_mean', 'S_12_mean', 'R_6_mean', 'S_13_mean', 'B_21_mean', 'D_69_mean', 'B_22_mean', 'D_70_mean', 'D_71_mean', 'D_72_mean', 'S_15_mean', 'B_23_mean', 'D_73_mean', 'P_4_mean', 'D_74_mean', 'D_75_mean', 'D_76_mean

C:\Users\HP\AppData\Local\Temp\ipykernel_7204\899398410.py:3: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print(df.columns)


In [12]:
all_importances = []
n_iterations = 3

defaults_df = df.filter(pl.col("target") == 1).collect()
non_defaults_df = df.filter(pl.col("target") == 0).collect()

for i in range(n_iterations):
    print(f"Iteration {i+1}/{n_iterations}...")
    
    sample = pl.concat([
        defaults_df.sample(n=100000, seed=i),
        non_defaults_df.sample(n=100000, seed=i)
    ])
    
   
    drop_cols = ["customer_ID", "target", "S_2"]
    X = sample.drop([c for c in drop_cols if c in sample.columns]).to_pandas()
    y = sample["target"].to_pandas()
    
    if i == 0:
        constant_cols = [col for col in X.columns if X[col].nunique() <= 1]
        print(f"Dropping {len(constant_cols)} constant columns.")
    
    X = X.drop(columns=constant_cols)
    
    selector = LGBMClassifier(
        n_estimators=250, 
        importance_type='gain', 
        random_state=i,
        device="gpu"
    )
    selector.fit(X, y)
    
    imp = pd.DataFrame({'feature': X.columns, f'gain_{i}': selector.feature_importances_})
    all_importances.append(imp.set_index('feature'))


final_rank = pd.concat(all_importances, axis=1).mean(axis=1).sort_values(ascending=False)

top_features_all = final_rank.head(50)
print("\n--- Top Features (Audit) ---")
print(top_features_all.head(10))

magic_25 = final_rank.head(25).index.tolist()
print("\nFINAL MAGIC 25 LIST:")
print(magic_25)

import json
with open('magic_25_features.json', 'w') as f:
    json.dump(magic_25, f)

Iteration 1/3...
Dropping 0 constant columns.
[LightGBM] [Info] Number of positive: 100000, number of negative: 100000
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 103033
[LightGBM] [Info] Number of data points in the train set: 200000, number of used features: 563
[LightGBM] [Info] Using GPU Device: Intel(R) UHD Graphics, Vendor: Intel(R) Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 303 dense feature groups (57.98 MB) transferred to GPU in 0.035760 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Iteration 2/3...
[LightGBM] [Info] Number of positive: 100000, number of negative: 100000
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 103049
[LightGBM] [Info] Number of data points in the train set: 200000, number of used features: 563
[Lig

In [35]:

print(top_25)


['P_2', 'B_3', 'B_11', 'S_3', 'D_42', 'B_9', 'D_45', 'B_37', 'D_43', 'R_1', 'B_4', 'D_61', 'D_46', 'D_50', 'B_6', 'D_44', 'R_27', 'B_23', 'B_2', 'B_38', 'D_41', 'D_51', 'D_56', 'D_48', 'P_3']
